In [26]:
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel , Field 
from langchain_core.output_parsers import PydanticOutputParser
from typing import Optional

In [2]:
load_dotenv()

True

In [3]:
loader  = PyPDFLoader('DEV RESUME.pdf')

In [4]:
doc = loader.load()

In [5]:
len(doc[0].page_content)

2509

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500 , 
    chunk_overlap = 0
)

In [7]:
splitted_doc = splitter.split_documents(doc)

In [8]:
for i in range(2):
    print(len(splitted_doc[i].page_content))

1482
1026


In [9]:
chroma_store = Chroma.from_documents(
    embedding = GoogleGenerativeAIEmbeddings(model ='gemini-embedding-2-preview') , 
    documents = splitted_doc
)

In [10]:
chroma_store.get(include = ['embeddings'])

{'ids': ['a561d9c9-d2a8-44ab-8191-da77ecb74bc4',
  '4fb27282-0a48-4f1a-9fa3-8ea61450923c'],
 'embeddings': array([[-0.01194284,  0.02228644, -0.00547701, ..., -0.00764944,
         -0.00436299,  0.00790595],
        [ 0.00438691,  0.0199843 ,  0.00965814, ..., -0.01257239,
         -0.00351297, -0.00623261]], shape=(2, 3072)),
 'documents': None,
 'uris': None,
 'included': ['embeddings'],
 'data': None,
 'metadatas': None}

In [19]:
retriver = chroma_store.as_retriever(search_kwargs={'k': 1})

In [20]:
query = 'What is my eduction'

In [21]:
retrive_doc = retriver.invoke(query)

In [28]:
class output(BaseModel):

    Education : str = Field(description='Just extract my education from the given text') 

    Education_year: int = Field(description="Extract my Engineering Year")

    git_hub : Optional[str] = Field(description='Extract the given github ID from the given Text')

In [ ]:
pydantic_parser = PydanticOutputParser(pydantic_object=output)

In [30]:
prompt = PromptTemplate(
    template='Extract the Education , Education Year and a if possible the Github link from the following text  \n{text} \n {format_instruction}' , 
    input_variables=['text'] , 
    partial_variables={'format_instruction' : pydantic_parser.get_format_instructions()}
)

In [31]:
model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [32]:
chain = prompt | model | pydantic_parser

In [35]:
text = retrive_doc[0].page_content
text

'Data Scientist April 2025 – Present\nMicroSpectra Software Technologies Pvt. Ltd. Shegaon, India\n•Worked on end-to-end machine learning pipelines including data preprocessing, feature engineering, and model training\n•Developed and evaluated ML and deep learning models for real-world business use cases\n•Implemented NLP solutions including text classiﬁcation and semantic similarity tasks\n•Focused on building scalable and production-ready models using Python and TensorFlow\nEDUCATION\nPune University Pune, India\nBachelor of Engineering (Mechanical Engineering) 2024 - 2027 Expected\n•Currently focusing on Data Science, Machine Learning, and Artiﬁcial Intelligence\nROLES & RESPONSIBILITIES\n•Build and train deep learning models using TensorFlow and Keras\n•Design end-to-end ML pipelines from raw data to model evaluation\n•Develop CNN-based models for image classiﬁcation and feature extraction\n•Implement NLP systems using embeddings, transformers, and sentence similarity\n•Fine-tune p

In [36]:
result = chain.invoke({'text' : text})

In [39]:
print(result.Education)
print(result.Education_year)
print(result.git_hub)

Bachelor of Engineering (Mechanical Engineering)
2027
None
